In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!nvidia-smi

Thu Aug 20 02:06:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
cuda_code=r'''
#include <cuda.h>
#include <cuda_runtime.h>
#include <torch/extension.h>

__global__ void vector_add_kernel(const float* A, const float* B, float* C, int N){
    
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i<N) {
        C[i] = A[i]+B[i];
    }
}

void vector_add_cuda(torch::Tensor A, torch::Tensor B, torch::Tensor C){
    int N = A.numel();
    int threads=256;
    int blocks=(N+threads-1)/threads;
    
    vector_add_kernel<<<blocks, threads>>>(A.data_ptr<float>(), B.data_ptr<float>(), C.data_ptr<float>(), N);
}
'''

with open("vector_add.cu", "w") as f:
    f.write(cuda_code)

In [5]:
cpp_code=r'''
#include <torch/extension.h>

void vector_add_cuda(torch::Tensor A, torch::Tensor B, torch::Tensor C);

void vector_add(torch::Tensor A, torch::Tensor B, torch::Tensor C){
    vector_add_cuda(A, B, C);
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m){
    m.def("vector_add", &vector_add, "Vector addition CUDA");
}
'''

with open("vector_add.cpp", "w") as f:
    f.write(cpp_code)

In [6]:
%ls

vector_add.cpp  vector_add.cu


In [7]:
from torch.utils.cpp_extension import load
vector_add_cuda=load(name="vector_add_cuda", 
                     sources=["vector_add.cu", "vector_add.cpp"], 
                     verbose=True)

print("Cuda compiled successfully!")

[1/3] c++ -MMD -MF vector_add.o.d -DTORCH_EXTENSION_NAME=vector_add_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -fPIC -std=c++17 -c /kaggle/working/vector_add.cpp -o vector_add.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output vector_add.cuda.o.d -DTORCH_EXTENSION_NAME=vector_add_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,co

# Testing

In [8]:
import torch

print("CUDA: ", torch.cuda.is_available())
print("GPU: ", torch.cuda.get_device_name())

CUDA:  True
GPU:  Tesla T4


In [9]:
N=1_000_000

A=torch.ones(N, device="cuda", dtype=torch.float32)
B=torch.ones(N, device="cuda", dtype=torch.float32)
C=torch.empty(N, device="cuda", dtype=torch.float32)

In [10]:
vector_add_cuda.vector_add(A, B, C)

In [11]:
print(C)

tensor([2., 2., 2.,  ..., 2., 2., 2.], device='cuda:0')


In [12]:
ans=A+B

assert torch.allclose(C, ans)

print("Answer Correct!!!")

Answer Correct!!!
